In [0]:
# ============================================================

# GOLD – dim_ticket_type

# Grain: (source_user_id, project, ticket_type_name)

# ============================================================

from pyspark.sql import functions as F

from pyspark.sql.window import Window

# ------------------------------------------------------------

# CONFIG

# ------------------------------------------------------------

SILVER_TABLE = "workspace.slainte_silver.easyvista_tickets_silver_demo"

GOLD_DB = "workspace.slainte_gold"

DIM_TTYPE = f"{GOLD_DB}.dim_ticket_type"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

# ------------------------------------------------------------

# 1️⃣ READ SILVER

# ------------------------------------------------------------

df_src = spark.table(SILVER_TABLE)

# ------------------------------------------------------------

# 2️⃣ BASE EXTRACTION

# ------------------------------------------------------------

df_ttype_base = (

    df_src

    .select(

        F.col("source_user_id"),

        F.col("project"),

        F.trim(F.lower(F.col("ticket_type"))).alias("ticket_type_name")

    )

    .filter(

        F.col("source_user_id").isNotNull() &

        F.col("project").isNotNull() &

        F.col("ticket_type_name").isNotNull()

    )

    .distinct()

)

# ------------------------------------------------------------

# 3️⃣ SURROGATE KEY

# ------------------------------------------------------------

window = Window.partitionBy(

    "source_user_id", "project"

).orderBy("ticket_type_name")

df_dim_ticket_type = (

    df_ttype_base

    .withColumn("ticket_type_id", F.row_number().over(window))

    .withColumn("created_at", F.current_timestamp())

    .select(

        "ticket_type_id",

        "source_user_id",

        "project",

        "ticket_type_name"

    )

)

# ------------------------------------------------------------

# 4️⃣ WRITE GOLD

# ------------------------------------------------------------

df_dim_ticket_type.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_TTYPE)

# ------------------------------------------------------------

# 5️⃣ PREVIEW

# ------------------------------------------------------------

print("✅ dim_ticket_type created successfully")

display(

    spark.table(DIM_TTYPE)

         .orderBy("source_user_id", "project", "ticket_type_id")

)
 